In [19]:
!pip install chromadb sentence-transformers tavily-python


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import glob
import chromadb
from sentence_transformers import SentenceTransformer

c:\Users\rachelle.l.macaraig\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
files = glob.glob("data/*.json")

raw_data = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        content = json.load(f)

        # handle BOTH single dict and list
        if isinstance(content, list):
            raw_data.extend(content)
        else:
            raw_data.append(content)

print("Total records:", len(raw_data))

Total records: 3


In [11]:
documents = []

for game in raw_data:
    text = f"""
    Title: {game.get('title', '')}
    Genre: {game.get('genre', '')}
    Platform: {game.get('platform', '')}
    Description: {game.get('description', '')}
    """
    documents.append(text.strip())

print("Processed documents:", len(documents))

Processed documents: 3


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\rachelle.l.macaraig\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rachelle.l.macaraig\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:0

In [13]:
embeddings = model.encode(documents)

print("Embeddings created:", len(embeddings))

Embeddings created: 3


In [15]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="games_collection"
)

In [16]:
for i, doc in enumerate(documents):
    collection.add(
        ids=[str(i)],
        documents=[doc],
        embeddings=[embeddings[i].tolist()]
    )

print("Data stored in ChromaDB")

Data stored in ChromaDB


In [17]:
query = "open world adventure game with exploration"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

results["documents"]

[['Title: Pokemon Red\n    Genre: RPG\n    Platform: Game Boy\n    Description:',
  'Title: God of War Ragnarok\n    Genre: Action\n    Platform: PS4, PS5\n    Description:',
  'Title: FIFA 21\n    Genre: Sports\n    Platform: PC, PS4, Xbox One\n    Description:']]

In [18]:
print("\n🔍 Semantic Search Results:\n")

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i+1}")
    print(doc)
    print("-" * 50)


🔍 Semantic Search Results:

Result 1
Title: Pokemon Red
    Genre: RPG
    Platform: Game Boy
    Description:
--------------------------------------------------
Result 2
Title: God of War Ragnarok
    Genre: Action
    Platform: PS4, PS5
    Description:
--------------------------------------------------
Result 3
Title: FIFA 21
    Genre: Sports
    Platform: PC, PS4, Xbox One
    Description:
--------------------------------------------------


## ✅ Project Summary

This notebook:
- Loads JSON game data
- Processes and formats text for embeddings
- Generates embeddings using SentenceTransformers
- Stores data in a persistent ChromaDB vector database
- Demonstrates semantic search using query embeddings